In [ ]:
import numpy as np
import itertools
import time
from raiselab.core import dvqe


# ============================================================
# USER SETTINGS
# ============================================================

n = 12
mode = "distributed"           # "monolithic" or "distributed"
qpu_qubit_config = [6, 5,5]      # must sum to at least n for distributed mode

depth = 2
init_type = 2                  # 1=random, 2=Black Hole, 3=GWO, 4=ABC

lr = 0.1
max_iters = 80
rel_tol = 1e-4

num_shots = 64
final_shots = 1000

warm_start_population = 3
warm_start_iters = 4
warm_start_shots = 128

seed = 100
density = 0.35

verbose = False               # False hides training iterations


# ============================================================
# BRUTE-FORCE EXACT QUBO SOLVER
# ============================================================

def brute_force_qubo(Q, q_linear):
    n = len(q_linear)
    best_cost = float("inf")
    best_solutions = []

    for bits in itertools.product([0, 1], repeat=n):
        z = np.array(bits, dtype=int)
        cost = float(z.T @ Q @ z + q_linear @ z)

        if cost < best_cost:
            best_cost = cost
            best_solutions = [z.copy()]
        elif abs(cost - best_cost) < 1e-9:
            best_solutions.append(z.copy())

    return best_solutions[0], best_cost, best_solutions


# ============================================================
# RANDOM QUBO GENERATOR WITH UNIQUE EXACT SOLUTION
# ============================================================

def create_random_qubo_unique_solution(
    n,
    seed=1,
    density=0.35,
    max_tries=500,
    value_low=-5,
    value_high=6
):
    for attempt in range(max_tries):
        rng = np.random.default_rng(seed + attempt)

        Q = rng.integers(value_low, value_high, size=(n, n)).astype(float)

        Q = np.triu(Q)
        Q = Q + Q.T - np.diag(np.diag(Q))

        mask = (rng.random((n, n)) < density).astype(float)
        mask = np.triu(mask)
        mask = mask + mask.T - np.diag(np.diag(mask))

        Q = Q * mask

        q_linear = rng.integers(value_low, value_high, size=n).astype(float)

        z_exact, cost_exact, all_best = brute_force_qubo(Q, q_linear)

        if len(all_best) == 1:
            print(f"Unique QUBO found at attempt {attempt + 1}")
            return Q, q_linear, z_exact, cost_exact

    raise RuntimeError("Could not find a QUBO with unique solution.")


# ============================================================
# RUN TEST
# ============================================================

print("=" * 70)
print("DVQE comparison test")
print(f"mode = {mode}")
print(f"n = {n}")
print(f"depth = {depth}")
print(f"init_type = {init_type}")
print(f"qpu_qubit_config = {qpu_qubit_config}")
print("=" * 70)

Q, q_linear, z_exact, cost_exact = create_random_qubo_unique_solution(
    n=n,
    seed=seed,
    density=density
)

print("\nExact solution:", z_exact)
print("Exact cost:", cost_exact)

start = time.time()

z_dvqe, final_circuit, hist = dvqe(
    mode=mode,
    Q=Q,
    q_linear=q_linear,
    init_type=init_type,
    depth=depth,
    lr=lr,
    max_iters=max_iters,
    qpu_qubit_config=qpu_qubit_config,
    rel_tol=rel_tol,
    num_shots=num_shots,
    final_shots=final_shots,
    warm_start_population=warm_start_population,
    warm_start_iters=warm_start_iters,
    warm_start_shots=warm_start_shots,
    seed=seed,
    verbose=verbose
)

elapsed = time.time() - start

cost_dvqe = float(z_dvqe.T @ Q @ z_dvqe + q_linear @ z_dvqe)
gap = cost_dvqe - cost_exact
exact_match = np.array_equal(z_dvqe, z_exact)

print("\n" + "-" * 70)
print("RESULT SUMMARY")
print("-" * 70)
print("Exact solution:       ", z_exact)
print("DVQE solution:        ", z_dvqe)
print("Exact cost:           ", cost_exact)
print("DVQE cost:            ", cost_dvqe)
print("Optimality gap:       ", gap)
print("Exact match:          ", exact_match)
print("Time:                 ", elapsed, "seconds")
print("-" * 70)

DVQE comparison test
mode = distributed
n = 12
depth = 2
init_type = 2
qpu_qubit_config = [6, 5, 5]
Unique QUBO found at attempt 1

Exact solution: [0 0 1 0 0 1 0 1 0 0 1 0]
Exact cost: -32.0
